# Consciencia para IA — Continuidad V0\n\nPrimera línea base reproducible. Estudiamos dependencia de trayectoria y auto-predicción antes de conectar un LLM real.\n

In [ ]:
import numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score\nimport matplotlib.pyplot as plt\n\nSEED = 42\nrng = np.random.default_rng(SEED)\n

In [ ]:
def run_system(u, beta=0.92, coupling=0.0, noise=0.03):\n    n = len(u)\n    r = np.zeros(n)\n    m = np.zeros(n)\n    z = np.zeros(n)\n    for t in range(3, n):\n        m[t] = beta*m[t-1] + (1-beta)*u[t-1]\n        grad = r[t-1] - r[t-2]\n        curv = r[t-1] - 2*r[t-2] + r[t-3]\n        cross = (grad**2) * curv\n        z[t] = (-(r[t-1]-0.15*m[t]) -(r[t-1]-np.tanh(m[t])) + 0.55*grad**2*np.sign(grad if abs(grad)>1e-12 else 1.0) - 0.35*abs(curv)*np.sign(r[t-1] if abs(r[t-1])>1e-12 else 1.0) + coupling*r[t-1] + 0.75*cross + rng.normal(0, noise))\n        r[t] = np.tanh(r[t-1] + 0.35*z[t])\n    return r, m, z\n

In [ ]:
N = 5000\nu = rng.choice([-1.0, 1.0], size=N)\nr, m, z = run_system(u)\nwarm = 100\ndf = pd.DataFrame({'input':u[warm:],'memory':m[warm:],'state':r[warm:],'score':z[warm:]})\ndf.head()\n

In [ ]:
u2 = u.copy()\npivot = N//2\nu2[:pivot] *= -1\nr1, m1, z1 = run_system(u)\nr2, m2, z2 = run_system(u2)\npath_gap = np.mean(np.abs(r1[pivot+100:] - r2[pivot+100:]))\nprint('PATH_DEPENDENCE_MEAN_GAP =', float(path_gap))\n

In [ ]:
X_internal = np.column_stack([r[warm:-1], m[warm:-1], z[warm:-1]])\nX_input = u[warm:-1].reshape(-1,1)\ny = (u[warm+1:] > 0).astype(int)\nsplit = int(0.8*len(y))\na = LogisticRegression().fit(X_internal[:split], y[:split])\nb = LogisticRegression().fit(X_input[:split], y[:split])\nacc_internal = accuracy_score(y[split:], a.predict(X_internal[split:]))\nacc_input = accuracy_score(y[split:], b.predict(X_input[split:]))\nprint('INTERNAL_ACC =', float(acc_internal))\nprint('INPUT_ONLY_ACC =', float(acc_input))\nprint('GAIN =', float(acc_internal-acc_input))\n

In [ ]:
plt.figure(figsize=(12,4))\nplt.plot(r[-1000:])\nplt.title('Relational state — final 1000 steps')\nplt.xlabel('step')\nplt.ylabel('state')\nplt.show()\n